In [282]:
import re

import pandas as pd

In [283]:
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_clean.csv'
df = pd.read_csv(file_name)

In [284]:
# Map the bill status sub-types to main types - {Passed, Failed, Pending, Vetoed}
bill_status_map = {
    "Enacted": "Passed",
    "Failed": "Failed",
    "Failed - Adjourned": "Failed",
    "Adopted": "Passed",
    "Pending": "Pending",
    "Pending - Carryover": "Pending",
    "To Governor": "Pending",
    "Override Pending": "Pending",
    "Vetoed": "Vetoed"
}
df["status_type"] = df.status.map(bill_status_map)

# Outcome flags
df["outcome_final"] = df.status_type != "Pending"
df["status_stale"] = (df.status_type == "Pending") & (df.page_year < df.page_year.max())

In [285]:
# Map the audience from the bill_number to the session's chamber
bill_audience_map = {'A': 'Assembly Bill',
                     'ACR': 'Assembly Concurrent Resolution',
                     'AJR': 'Assembly Joint Resolution',
                     'AR': 'Assembly Resolution',
                     'B': 'Bill',
                     'BDR': 'Bill Draft Request',
                     'D': 'Draft Bill',
                     'H': 'House Bill',
                     'HCR': 'House Concurrent Resolution',
                     'HD': 'House Docket',
                     'HJM': 'House Joint Memorial',
                     'HJR': 'House Joint Resolution',
                     'HM': 'House Memorial',
                     'HR': 'House Resolution',
                     'HSB': 'House Study Bill ',
                     'L': 'Lesislative Bill',
                     'LCR': 'Legislative Council Resolution',
                     'LR': 'Legislative Reference',
                     'S': 'Senate Bill',
                     'SCR': 'Senate Concurrent Resolution',
                     'SD': 'Senate Docket',
                     'SJR': 'Senate Joint Resolution',
                     'SM': 'Senate Memorial',
                     'SPB': 'Senate Bill',
                     'SR': 'Senate Resolution',
                     'SSB': 'Senate Study Bill',
                     'WS': 'Senate Bill'}

# Split 'bill_number' into chamber prefix and bill number into columns
df[['chamber_prefix', 'bill_number']] = df.bill_number.str.split(
    ' ', n=1, expand=True)

# Clean bill numbers and create an integer type column
df.bill_number = df.bill_number.str.extract(r"^(\d+)")[0].astype(int)

# Get full chamber name from the prefix
df["chamber_name"] = df.chamber_prefix.map(bill_audience_map)


In [286]:
document_types = ['Resolution', 'Study Bill',
                  'Memorial', 'Docket', 'Draft', 'Reference']

pattern = f"(?i)({'|'.join(document_types)})"

# Extract matching document type, will "Bill" as default match
df['document_type'] =  df['chamber_name'].str.extract(pattern, expand=False).fillna('Bill')

In [287]:
# Chamber (from the bill prefix). "Other" = drafts, legislative references, DC/territory bills
df["chamber"] = df.chamber_prefix.str[0].map({"H": "House", "S": "Senate", "A": "Assembly"}).fillna("Other")
df.loc[df.chamber_prefix == "WS", "chamber"] = "Senate"
df["is_lower_chamber"] = df.chamber.isin(["House", "Assembly"])

#  Correct resolutions typed as bills (RI/SC use bill-style prefixes; their summary starts with a "(Resolution)" tag)
res_tag = r"^\((?:Senate |House )?Resolution\)\s*"
is_tagged = df.summary.str.contains(res_tag, na=False)
df.loc[is_tagged, "chamber_name"] = df.loc[is_tagged, "chamber_name"].str.replace("Bill", "Resolution")
df["summary"] = df.summary.str.replace(res_tag, "", regex=True)

In [288]:
# Add year each session took place and whether it was a special session
session = pd.to_numeric(df.bill_id.str.extract(r"^[A-Z]{2}(\d{4})")[0])
df["session_year"] = session.fillna(df.page_year).astype(int)

df["is_election_year"] = df.session_year % 2 == 0

df["is_enacted_bill"] = (df.status_type == "Passed") & (df.document_type == "Bill")
df["is_enacted_bill"] = (df.status_type == "Passed") & (df.document_type == "Bill")
state_session = (df.groupby(["state", "session_year"])
                   .agg(state_session_bill_count=("bill_id", "size"), enacted=("is_enacted_bill", "sum"))
                   .reset_index().sort_values(["state", "session_year"]))
state_session["state_prior_bills"] = state_session.groupby("state").state_session_bill_count.cumsum() - state_session.state_session_bill_count
state_session["state_prior_enacted"] = state_session.groupby("state").enacted.cumsum() - state_session.enacted

# Get # of companion bills (same bill introduced through different legislative bodies)
df["companion_bill_count"] = df.groupby(
    ["state", "session_year", "title", "summary"], dropna=False).bill_id.transform("size")

In [289]:
# Encode different types of legislation
df["is_resolution"] = df.chamber_name.str.contains("Resolution")
df["is_joint_resolution"] = df.chamber_name.str.contains("Joint Resolution")
df["is_concurrent_resolution"] = df.chamber_name.str.contains(
    "Concurrent Resolution")
df["is_simple_resolution"] = df.chamber_prefix.isin(["HR", "SR", "AR"])

In [290]:
# Gets the type of each state/area - {State, Territory, District}
df['jurisdiction_type'] = df.state.map(
    lambda s: "Territory" if s in {"PR", "VI", "GU", "AS", "MP"}
    else "District" if s == "DC" else "State")

region_map = {
    **dict.fromkeys(["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"], "Northeast"),
    **dict.fromkeys(["IL", "IN", "MI", "OH", "WI", "IA", "KS", "MN", "MO", "NE", "ND", "SD"], "Midwest"),
    **dict.fromkeys(["DE", "DC", "FL", "GA", "MD", "NC", "SC", "VA", "WV", "AL", "KY", "MS", "TN",
                     "AR", "LA", "OK", "TX"], "South"),
    **dict.fromkeys(["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY", "AK", "CA", "HI", "OR", "WA"], "West"),
    **dict.fromkeys(["PR", "VI", "GU", "AS", "MP"], "Territory"),
}
df["census_region"] = df.state.map(region_map)
assert df.census_region.notna().all(), df.loc[df.census_region.isna(), "state"].unique()

In [291]:
def any_of(names):
    return df[[f"category_{n}" for n in names]].max(axis=1)

# Now we can turn the multi-label category list into one-hot-encoded columns
df.category = df.category.fillna("").str.lower()
df.category = df.category.str.replace(' ', '_')

In [292]:
# One hot encoding categories from multi-label text
# Split strings by separator, strip whitespace, and create dummy variables
dummies = df.category.fillna("").str.get_dummies(sep=";")
dummies = dummies.rename(columns=lambda c: "category_" +
                         re.sub(r"\W+", "_", c.lower()))

# Total number of categories
df["num_categories"] = dummies.sum(axis=1)

# Add new category columns to existing dataset
df = pd.concat([df, dummies], axis=1)

# Whether a bill was for study-use only
df["studies_only"] = (dummies.category_studies == 1) & (df.num_categories == 1)

In [293]:
# Category groups (from the one-hot flags above) and a flag for bills NCSL left uncategorized.
df["catgroup_public_sector"] = any_of(["government_use", "judicial_use", "appropriations", "audit"])
df["catgroup_regulatory"] = any_of(["private_sector", "notification", "private_right_of_action",
                                    "impact_assessment", "oversight_governance", "responsible_use"])
df["catgroup_rights_safety"] = any_of(["criminal_use", "child_pornography", "elections", "personhood"])
df["catgroup_sector_specific"] = any_of(["health_use", "education_use", "education_training", "housing",
                                         "energy", "effect_on_labor_employment", "cybersecurity", "taxes"])

# 'Education Use' only exists on the 2024+ pages, so catgroup_education works across years
df["catgroup_education"] = any_of(["education_use", "education_training"])

In [294]:
# Replace curly quotes in 'title' and 'summary' columns
quotes = str.maketrans({"\u2018": "'", "\u2019": "'",
                       "\u201c": '"', "\u201d": '"'})
for c in ["title", "summary"]:
    df[c] = df[c].map(lambda s: s.translate(quotes)
                      if isinstance(s, str) else s)

In [295]:
# Final data quality checks
assert df.bill_id.is_unique
assert (df.session_year <= df.page_year).all()
assert df.state.notna().all() and df.bill_number.notna().all()
# assert (df.isnull().sum().sum() == 0)
# print(df.isnull().sum()[lambda s: s > 0])

df = df.drop(columns=["category", "category_", "chamber_name", "chamber_name", 
                      "page_year"])

# Export final csv to dataset
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_final.csv'
df.to_csv(file_name, index=False)

# View final columns for 2018 - 2025 bills dataset
print(df.columns)

Index(['bill_id', 'bill_number', 'state', 'status', 'title', 'summary',
       'status_type', 'outcome_final', 'status_stale', 'chamber_prefix',
       'document_type', 'chamber', 'is_lower_chamber', 'session_year',
       'is_election_year', 'is_enacted_bill', 'companion_bill_count',
       'is_resolution', 'is_joint_resolution', 'is_concurrent_resolution',
       'is_simple_resolution', 'jurisdiction_type', 'census_region',
       'num_categories', 'category_appropriations', 'category_audit',
       'category_child_pornography', 'category_criminal_use',
       'category_cybersecurity', 'category_education_training',
       'category_education_use', 'category_effect_on_labor_employment',
       'category_elections', 'category_energy', 'category_government_use',
       'category_health_use', 'category_housing', 'category_impact_assessment',
       'category_judicial_use', 'category_notification',
       'category_oversight_governance', 'category_personhood',
       'category_private_ri